In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DataCollatorForTokenClassification
import numpy as np
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import json
from pathlib import Path

In [45]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

def encode_tags(example, mapping):
    example["ner_tags"] = [mapping[x] for x in example["iob_tags"]]
    return example

In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
seqeval = evaluate.load("seqeval")
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

In [19]:
dataset_path = Path("data/rudeuft_hf/")

data = []
with open(dataset_path/"dev.json") as f:
    for l in f.readlines():
        data.append(json.loads(l)) 
        
with open(dataset_path/"dev.json", 'w') as f:
    json.dump(data, f)
    
data = []
with open(dataset_path/"train.json") as f:
    for l in f.readlines():
        data.append(json.loads(l)) 
        
with open(dataset_path/"train.json", 'w') as f:
    json.dump(data, f)

In [21]:
dataset = load_dataset("json", data_files={"train": dataset_path/"train.json", "test": dataset_path/"dev.json"})

In [54]:
label_list = list({x for y in dataset["train"]["iob_tags"] for x in y})
id2label = {x:y for x,y in enumerate(label_list)}
label2id = {y:x for x,y in id2label.items()}

In [42]:
dataset = dataset.map(lambda x: encode_tags(x, label2id), batch_size=True)

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [43]:
dataset["train"][0]

{'text': '5 .',
 'definition_label': 0,
 'source_file': 'data/source_txt/t1_biology_0_0.deft',
 'tokens': ['5', '.'],
 'iob_tags': ['O', 'O'],
 'spans': [],
 'relations': [],
 'text_rus': '',
 'ner_tags': [19, 19]}

In [46]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [47]:
model = AutoModelForTokenClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=len(id2label), id2label=id2label, label2id=label2id)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [51]:
training_args = TrainingArguments(
    output_dir="deft_model",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

In [55]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Parameters must be of builtin type (Transformers_6/accelerator_config[AcceleratorConfig])


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.365500,0.450137,0.449878,0.538012,0.490013,0.853114
2,0.330300,0.472854,0.463804,0.552632,0.504336,0.853828


/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
████████████████████████████▊  99% | 755.00/760.69 MB [00:08<00:00, 85.41MB/s]: /usr/local/lib/python3.10/dist-packages/tqdm/std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
█████████████████████████████ 100% | 760.69/760.69 MB [00:09<00:00, 78.55MB/s]: 
/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
███████████████████████████▉ 100% | 760.00/760.69 MB [00:04<00:00, 160.98MB/s]: /usr/local/lib/python3.10/dist-packages/tq

TrainOutput(global_step=1512, training_loss=0.3482400804915756, metrics={'train_runtime': 77.5696, 'train_samples_per_second': 623.543, 'train_steps_per_second': 19.492, 'total_flos': 816659586523008.0, 'train_loss': 0.3482400804915756, 'epoch': 2.0})